# RL-11 : POMDP - Partial Observability et Belief Tracking

**Notebook : 11** | **Duree** : 45-50 min | **Kernel** : .NET C#

Dans les notebooks précédents, l'agent connaissait toujours l'etat exact de l'environnement.
Mais dans la realite, un robot ne voit pas parfaitement, un capteur est bruite, et un agent
negociateur ne connait pas les cartes de son adversaire.

Les **POMDP** (Partially Observable Markov Decision Processes) modelisent cette incertitude
observationnelle. L'agent ne voit plus l'etat $s$, mais une **observation** $o$ qui depend
probabilistement de $s$.

Ce notebook explore :

1. **Le Tiger Problem** : un POMDP classique (Cassandra et al., 1994)
2. **Politiques hand-crafted** : comment l'observation bruitee affecte les decisions
3. **Belief tracking** : maintenir une distribution de probabilite sur les etats caches
4. **Q-MDP approximation** : utiliser le belief state dans un cadre Q-learning

**Prerequis** : Notebooks 5 (MDP, Q-Learning) et 6 (DQN, politique epsilon-greedy).

## 1. Le Tiger Problem

Le **Tiger Problem** (Cassandra, Littman & Kaelbling, 1994) est un POMDP emblematique :

- **2 etats** : le tigre est derriere la porte gauche (0) ou droite (1)
- **3 actions** : ouvrir gauche (0), ouvrir droite (1), ecouter (2)
- **2 observations** : grognement gauche (0) ou droite (1)
- **Ecouter** : observation correcte a 85%, cout -1
- **Ouvrir porte tigre** : recompense -100, **ouvrir porte tresor** : +10
- Après ouverture, le problème **reset** (nouvelle position aleatoire du tigre)

La cle : l'agent ne voit jamais directement la position du tigre. Il doit **inferer** cette
position a partir des observations successives.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

const int SEED = 42;
const double ALPHA = 0.1, GAMMA = 0.95, EPSILON = 0.1;
const int MAX_STEPS = 20;
const double P_CORRECT = 0.85; // precision de l'observation en ecoutant

void VerifierProche(string nom, double mesure, double attendu, double tolerance)
{
    double ecart = Math.Abs(mesure - attendu);
    Console.WriteLine(FormattableString.Invariant($"{nom} : calcule={mesure:F4}, attendu={attendu:F4}, ecart={ecart:F4}, tolerance={tolerance:F4} : {(ecart <= tolerance ? "CONFORME" : "ECART")}"));
    if (double.IsNaN(mesure) || ecart > tolerance)
        throw new InvalidOperationException($"Assertion echouee : {nom}");
}

class TigerPOMDP
{
    public int n_states = 2;
    public int n_actions = 3;
    public int n_observations = 2;
    public double p_correct;
    public int tiger_pos;
    private bool just_listened;
    // Generateur propre a l'environnement (seede) : toutes les sorties
    // chiffrees du carnet viennent de ce flux, elles doivent etre reproductibles.
    private Random rng;

    public TigerPOMDP(double p_correct = 0.85, int seed = 42)
    {
        this.p_correct = p_correct;
        rng = new Random(seed);
        tiger_pos = 0;
        just_listened = false;
    }

    public void reset()
    {
        tiger_pos = rng.Next(2);
        just_listened = false;
    }

    private int obs()
    {
        if (just_listened)
            return rng.NextDouble() < p_correct ? tiger_pos : 1 - tiger_pos;
        return rng.Next(2);
    }

    public (int observation, double reward, bool done) step(int action)
    {
        if (action == 2) // listen
        {
            just_listened = true;
            return (obs(), -1.0, false);
        }
        else // open door
        {
            double reward = action == tiger_pos ? -100.0 : 10.0;
            tiger_pos = rng.Next(2);
            just_listened = true;
            return (obs(), reward, true);
        }
    }

    public int state => tiger_pos;
}

var env = new TigerPOMDP();
if (env.n_states != 2 || env.n_actions != 3 || env.n_observations != 2 || env.p_correct != 0.85)
    throw new InvalidOperationException("Contrat Tiger incorrect");
Console.WriteLine($"Tiger POMDP: {env.n_states} etats, {env.n_actions} actions, {env.n_observations} observations");
Console.WriteLine(FormattableString.Invariant($"Precision ecoute: {env.p_correct * 100:F0}%"));
Console.WriteLine("Cout ecoute: -1, Tigre: -100, Tresor: +10");

Tiger POMDP: 2 etats, 3 actions, 2 observations


Precision ecoute: 85%


Cout ecoute: -1, Tigre: -100, Tresor: +10


**Lecture chiffree — la structure de couts.** Trois lignes posent le contrat : `Precision ecoute: 85%`, `Cout ecoute: -1, Tigre: -100, Tresor: +10`. Deux asymmetries gouvernent tout le notebook. (1) L'echelle : se tromper de porte coute dix fois le gain de la reussite (+10 contre -100) — l'agent doit etre bien plus sur de lui que "deux chances sur trois". (2) L'information se paie : `reset()` ne revele rien (variante canonique du Tiger Problem de Cassandra et al., 1994), la seule source d'information est l'action `listen`, facturee -1 chacune. Ce contrat fixe deja la limite des politiques aveugles : ouvrir sans ecouter vaut exactement (10 - 100)/2 = **-45**, quelle que soit la porte choisie — la section suivante mesure precisement cela.

## 2. Politiques baselines

Comparons d'abord des politiques simples pour comprendre l'espace des solutions :

| Politique | Stratégie |
|-----------|-----------|
| **Random** | Action aleatoire a chaque pas |
| **Open immediately** | Ouvre une porte au hasard, sans jamais ecouter (aveugle) |
| **Listen x1** | Ecoute une fois, puis ouvre la porte opposee |
| **Listen x2** | Ecoute deux fois (vote majoritaire), puis ouvre |

In [2]:
(double mean, double[] returns) random_agent(TigerPOMDP environnement, int n_episodes, int seed = 0)
{
    var rng = new Random(seed);
    var returns = new double[n_episodes];
    for (int episode = 0; episode < n_episodes; episode++)
    {
        environnement.reset();
        double ret = 0.0;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            int action = rng.Next(3);
            var transition = environnement.step(action);
            ret += transition.reward;
            if (transition.done) break;
        }
        returns[episode] = ret;
    }
    return (returns.Average(), returns);
}

(double mean, double[] returns) open_immediately(TigerPOMDP environnement, int n_episodes, int seed = 0)
{
    var rng = new Random(seed);
    var returns = new double[n_episodes];
    for (int episode = 0; episode < n_episodes; episode++)
    {
        environnement.reset();
        int action = rng.Next(2); // aucune information : choix uniforme
        var transition = environnement.step(action);
        returns[episode] = transition.reward;
    }
    return (returns.Average(), returns);
}

(double mean, double[] returns) listen_then_open(TigerPOMDP environnement, int n_episodes, int n_listen = 2, int seed = 0)
{
    var rng = new Random(seed);
    var returns = new double[n_episodes];
    for (int episode = 0; episode < n_episodes; episode++)
    {
        environnement.reset();
        double[] belief = { 0.5, 0.5 };
        double ret = 0.0;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            int action;
            if (t < n_listen)
                action = 2; // listen
            else
                action = belief[0] <= belief[1] ? 0 : 1; // porte la moins probable

            var transition = environnement.step(action);
            ret += transition.reward;

            // Mise a jour bayesienne du belief -- le filtre lit la precision
            // REELLE du capteur (env.p_correct), pas le global P_CORRECT : a 0.85
            // c'est la meme valeur, des que le capteur varie le filtre reste
            // bien specifie (#25796, diagnostic po-2023 recable ici).
            if (action == 2)
            {
                double p0 = transition.observation == 0 ? environnement.p_correct : 1 - environnement.p_correct;
                double p1 = transition.observation == 1 ? environnement.p_correct : 1 - environnement.p_correct;
                belief[0] *= p0;
                belief[1] *= p1;
                double total = belief[0] + belief[1];
                belief[0] /= total;
                belief[1] /= total;
            }
            if (transition.done) break;
        }
        returns[episode] = ret;
    }
    return (returns.Average(), returns);
}

const int N_EP = 3000;
// Comparaison Monte Carlo : tolerance fixee avant les tirages, en points de retour.
const double TOL_BASELINES = 8.0;
Console.WriteLine(FormattableString.Invariant($"Tolerance des ecarts aux valeurs Python citees : {TOL_BASELINES:F1} points ; RNG C# distinct de NumPy."));
env = new TigerPOMDP();
Console.WriteLine("=== Politiques baselines ===\n");
var ret_rand = random_agent(env, N_EP, seed: SEED).mean;
var ret_imm = open_immediately(env, N_EP, seed: SEED).mean;
var ret_l1 = listen_then_open(env, N_EP, n_listen: 1, seed: SEED).mean;
var ret_l2 = listen_then_open(env, N_EP, n_listen: 2, seed: SEED).mean;
Console.WriteLine(FormattableString.Invariant($"Random:                {ret_rand:F1}"));
Console.WriteLine(FormattableString.Invariant($"Open immediately:      {ret_imm:F1}"));
Console.WriteLine(FormattableString.Invariant($"Listen x1 then open:   {ret_l1:F1}"));
Console.WriteLine(FormattableString.Invariant($"Listen x2 then open:   {ret_l2:F1}"));
Console.WriteLine("\nComparaison aux sorties Python citees :");
VerifierProche("Random", ret_rand, -45.9, TOL_BASELINES);
VerifierProche("Open immediately", ret_imm, -42.5, TOL_BASELINES);
VerifierProche("Listen x1", ret_l1, -8.1, TOL_BASELINES);
VerifierProche("Listen x2", ret_l2, -8.1, TOL_BASELINES);
VerifierProche("Esperance ouverture aveugle", (10.0 - 100.0) / 2.0, -45.0, 1e-12);
VerifierProche("Esperance une ecoute", P_CORRECT * 10.0 + (1.0 - P_CORRECT) * -100.0 - 1.0, -7.5, 1e-12);

Tolerance des ecarts aux valeurs Python citees : 8.0 points ; RNG C# distinct de NumPy.


=== Politiques baselines ===



Random:                -46.1


Open immediately:      -45.9


Listen x1 then open:   -9.1


Listen x2 then open:   -9.2


... (3 outputs left out)

Listen x1 : calcule=-9.0767, attendu=-8.1000, ecart=0.9767, tolerance=8.0000 : CONFORME


Listen x2 : calcule=-9.1967, attendu=-8.1000, ecart=1.0967, tolerance=8.0000 : CONFORME


Esperance ouverture aveugle : calcule=-45.0000, attendu=-45.0000, ecart=0.0000, tolerance=0.0000 : CONFORME


Esperance une ecoute : calcule=-7.5000, attendu=-7.5000, ecart=0.0000, tolerance=0.0000 : CONFORME


**Lecture chiffrée — les baselines.** Dans ce carnet C#, les retours sont -46.1 pour Random, -45.9 pour l'ouverture immédiate, -9.1 pour une écoute et -9.2 pour deux écoutes. Les sorties Python citées sont respectivement -45.9, -42.5, -8.1 et -8.1 : les générateurs étant distincts, les tirages diffèrent.

Ouvrir à l'aveugle vaut en espérance (10 - 100)/2 = **-45**. Une écoute correcte à 85 % donne une espérance de **-7.5**, coût compris. Avec deux observations contradictoires, le belief revient à 0.50 ; la deuxième écoute coûte néanmoins -1.

## 3. Belief Tracking (Filtre Bayesien)

L'agent maintient un **belief state** $b(s) = P(s | o_1, a_1, o_2, a_2, \ldots)$,
une distribution de probabilite sur les etats possibles.

### Mise a jour bayesienne

Après avoir pris l'action $a$ et observe $o$ :

$$b'(s) = \eta \cdot P(o | s, a) \sum_{s'} P(s | s', a) \cdot b(s')$$

ou $\eta$ est une constante de normalisation.

Dans le Tiger Problem, le belief se simplifie en $b = P(\text{tiger-left})$ :
- Si on ecoute et on entend growl-left : $b$ augmente
- Si on ecoute et on entend growl-right : $b$ diminue

In [3]:
double mise_a_jour_belief(double belief, int observation, double precision)
{
    double p0 = observation == 0 ? precision : 1.0 - precision;
    double p1 = observation == 1 ? precision : 1.0 - precision;
    double gauche = belief * p0;
    double droite = (1.0 - belief) * p1;
    return gauche / (gauche + droite);
}

void belief_update_demo()
{
    var environnement = new TigerPOMDP();
    environnement.reset();
    double belief = 0.5; // P(tiger-left)
    Console.WriteLine("=== Demonstration du Belief Tracking : tirage C# ===");
    Console.WriteLine($"Tigre reel: {(environnement.tiger_pos == 0 ? "gauche" : "droite")}");
    Console.WriteLine(FormattableString.Invariant($"Belief initial: P(tiger-left) = {belief:F2}\n"));
    for (int step = 0; step < 5; step++)
    {
        int action = 2; // listen
        var transition = environnement.step(action);
        // Bayesian update
        belief = mise_a_jour_belief(belief, transition.observation, P_CORRECT);
        Console.WriteLine(FormattableString.Invariant($"Step {step + 1}: obs={(transition.observation == 0 ? "growl-left" : "growl-right")} -> P(tiger-left) = {belief:F4}"));
        if (belief > 0.95)
        {
            Console.WriteLine("  -> Confiance elevee : tiger a gauche");
            break;
        }
        if (belief < 0.05)
        {
            Console.WriteLine("  -> Confiance elevee : tiger a droite");
            break;
        }
    }
    Console.WriteLine($"\nTigre reel: {(environnement.tiger_pos == 0 ? "gauche" : "droite")}");
    Console.WriteLine(FormattableString.Invariant($"Belief final: P(tiger-left) = {belief:F4}"));
}

belief_update_demo();

// Observations tirees par le jumeau Python, reprises comme donnees citees.
int[] observations_python = { 0, 1, 0, 0 };
double[] attendus_python = { 0.85, 0.50, 0.85, 0.9698 };
double belief_cite = 0.5;
Console.WriteLine("\nVerification de la trajectoire citee du jumeau Python :");
for (int i = 0; i < observations_python.Length; i++)
{
    belief_cite = mise_a_jour_belief(belief_cite, observations_python[i], P_CORRECT);
    VerifierProche($"Observation citee {i + 1}", belief_cite, attendus_python[i], 0.00005);
}

=== Demonstration du Belief Tracking : tirage C# ===


Tigre reel: droite


Belief initial: P(tiger-left) = 0.50



Step 1: obs=growl-right -> P(tiger-left) = 0.1500


Step 2: obs=growl-right -> P(tiger-left) = 0.0302


  -> Confiance elevee : tiger a droite


... (3 outputs left out)

Observation citee 1 : calcule=0.8500, attendu=0.8500, ecart=0.0000, tolerance=0.0001 : CONFORME


Observation citee 2 : calcule=0.5000, attendu=0.5000, ecart=0.0000, tolerance=0.0001 : CONFORME


Observation citee 3 : calcule=0.8500, attendu=0.8500, ecart=0.0000, tolerance=0.0001 : CONFORME


Observation citee 4 : calcule=0.9698, attendu=0.9698, ecart=0.0000, tolerance=0.0001 : CONFORME


**Lecture chiffrée — le filtre bayésien.** Dans le tirage C#, le tigre est à droite et deux observations à droite font passer la croyance de 0.50 à 0.1500 puis 0.0302 : le seuil de confiance est atteint après deux écoutes.

La trajectoire **citée du Python** est différente : gauche, droite, gauche, gauche ; elle donne 0.8500, 0.5000, 0.8500 puis 0.9698. La cellule précédente recalcule ces croyances à partir des observations citées. Une observation contradictoire peut ramener le belief vers 0.50 ; la confiance dépend du bilan des observations, pas simplement de leur nombre.

Le belief converge rapidement vers la bonne reponse (en general en 2-3 observations).
Mais la convergence n'est pas garantie : avec $P = 0.85$, une observation incorrecte peut
temporairement degrader le belief.

### Graphique d'evolution du belief

In [4]:
void plot_belief_evolution(int n_episodes = 20, int seed = 42)
{
    // Comme dans la source, chaque panneau recree un environnement de graine 42 ;
    // n_episodes et seed ne modifient pas les quatre trajectoires affichees.
    var rng = new Random(seed);
    var trajectoires = new List<double[]>();
    var positions = new List<int>();
    Console.WriteLine("Episode | Tigre | Ecoutes | P(tiger-left)");
    for (int idx = 0; idx < 4; idx++)
    {
        var environnement = new TigerPOMDP();
        environnement.reset();
        double belief = 0.5;
        var beliefs = new double[9];
        beliefs[0] = belief;
        for (int step = 0; step < 8; step++)
        {
            var transition = environnement.step(2); // listen
            belief = mise_a_jour_belief(belief, transition.observation, P_CORRECT);
            beliefs[step + 1] = belief;
        }
        trajectoires.Add(beliefs);
        positions.Add(environnement.tiger_pos);
        for (int step = 0; step < beliefs.Length; step++)
        {
            if (beliefs[step] < 0 || beliefs[step] > 1)
                throw new InvalidOperationException("Croyance hors de [0,1]");
            Console.WriteLine(FormattableString.Invariant($"{idx + 1,7} | {(environnement.tiger_pos == 0 ? "gauche" : "droite"),6} | {step,7} | {beliefs[step]:F8}"));
        }
    }

    // Le SVG utilise les memes valeurs que la table ci-dessus.
    var svg = new StringBuilder();
    svg.Append("<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='660' viewBox='0 0 1000 660' role='img' aria-label='Evolution du belief apres ecoutes successives'>");
    svg.Append("<rect width='1000' height='660' fill='white'/><text x='500' y='25' text-anchor='middle' font-size='20'>Evolution du belief apres ecoutes successives</text>");
    for (int idx = 0; idx < 4; idx++)
    {
        int x0 = 70 + (idx % 2) * 480;
        int y0 = 65 + (idx / 2) * 300;
        int width = 390, height = 220;
        double yTrue = y0 + height * (1.0 - (positions[idx] == 0 ? 1.0 : 0.0));
        svg.Append(FormattableString.Invariant($"<rect x='{x0}' y='{y0}' width='{width}' height='{height}' fill='none' stroke='black'/><line x1='{x0}' x2='{x0 + width}' y1='{y0 + height / 2}' y2='{y0 + height / 2}' stroke='gray' stroke-dasharray='6 4'/><line x1='{x0}' x2='{x0 + width}' y1='{yTrue:F2}' y2='{yTrue:F2}' stroke='green' stroke-dasharray='2 4'/>"));
        var points = new StringBuilder();
        for (int step = 0; step < 9; step++)
        {
            double x = x0 + width * step / 8.0;
            double y = y0 + height * (1.0 - trajectoires[idx][step]);
            points.Append(FormattableString.Invariant($"{x:F2},{y:F2} "));
        }
        svg.Append($"<polyline points='{points}' fill='none' stroke='blue' stroke-width='2'/>");
        for (int step = 0; step < 9; step++)
        {
            double x = x0 + width * step / 8.0;
            double y = y0 + height * (1.0 - trajectoires[idx][step]);
            svg.Append(FormattableString.Invariant($"<circle cx='{x:F2}' cy='{y:F2}' r='3' fill='blue'/>"));
        }
        svg.Append(FormattableString.Invariant($"<text x='{x0 + width / 2}' y='{y0 - 10}' text-anchor='middle'>Episode {idx + 1} (tigre {(positions[idx] == 0 ? "gauche" : "droite")})</text><text x='{x0 + width / 2}' y='{y0 + height + 24}' text-anchor='middle'>Ecoutes : 0 a 8</text><text x='{x0 - 8}' y='{y0 + 5}' text-anchor='end'>1.0</text><text x='{x0 - 8}' y='{y0 + height / 2 + 5}' text-anchor='end'>0.5</text><text x='{x0 - 8}' y='{y0 + height}' text-anchor='end'>0.0</text><text x='{x0 + 5}' y='{y0 + 18}' fill='green'>Tigre reel</text>"));
    }
    svg.Append("</svg>");
    display(svg.ToString(), "text/html");
}

plot_belief_evolution();

Episode | Tigre | Ecoutes | P(tiger-left)


      1 | droite |       0 | 0.50000000


      1 | droite |       1 | 0.15000000


      1 | droite |       2 | 0.03020134


      1 | droite |       3 | 0.00546559


      1 | droite |       4 | 0.00096888


... (28 outputs left out)

      4 | droite |       6 | 0.00003020


      4 | droite |       7 | 0.00000533


      4 | droite |       8 | 0.00000094


<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='660' viewBox='0 0 1000 660' role='img' aria-label='Evolution du belief apres ecoutes successives'><rect width='1000' height='660' fill='white'/><text x='500' y='25' text-anchor='middle' font-size='20'>Evolution du belief apres ecoutes successives</text><rect x='70' y='65' width='390' height='220' fill='none' stroke='black'/><line x1='70' x2='460' y1='175' y2='175' stroke='gray' stroke-dasharray='6 4'/><line x1='70' x2='460' y1='285.00' y2='285.00' stroke='green' stroke-dasharray='2 4'/><polyline points='70.00,175.00 118.75,252.00 167.50,278.36 216.25,283.80 265.00,284.79 313.75,284.96 362.50,284.99 411.25,285.00 460.00,285.00 ' fill='none' stroke='blue' stroke-width='2'/><circle cx='70.00' cy='175.00' r='3' fill='blue'/><circle cx='118.75' cy='252.00' r='3' fill='blue'/><circle cx='167.50' cy='278.36' r='3' fill='blue'/><circle cx='216.25' cy='283.80' r='3' fill='blue'/><circle cx='265.00' cy='284.79' r='3' fill='blue'/><circle cx='313.75' cy='284.96' r='3' fill='blue'/><circle cx='362.50' cy='284.99' r='3' fill='blue'/><circle cx='411.25' cy='285.00' r='3' fill='blue'/><circle cx='460.00' cy='285.00' r='3' fill='blue'/><text x='265' y='55' text-anchor='middle'>Episode 1 (tigre droite)</text><text x='265' y='309' text-anchor='middle'>Ecoutes : 0 a 8</text><text x='62' y='70' text-anchor='end'>1.0</text><text x='62' y='180' text-anchor='end'>0.5</text><text x='62' y='285' text-anchor='end'>0.0</text><text x='75' y='83' fill='green'>Tigre reel</text><rect x='550' y='65' width='390' height='220' fill='none' stroke='black'/><line x1='550' x2='940' y1='175' y2='175' stroke='gray' stroke-dasharray='6 4'/><line x1='550' x2='940' y1='285.00' y2='285.00' stroke='green' stroke-dasharray='2 4'/><polyline points='550.00,175.00 598.75,252.00 647.50,278.36 696.25,283.80 745.00,284.79 793.75,284.96 842.50,284.99 891.25,285.00 940.00,285.00 ' fill='none' stroke='blue' stroke-width='2'/><circle cx='550.00' cy='175.00' r='3' fill='blue'/><circle cx='598.75' cy='252.00' r='3' fill='blue'/><circle cx='647.50' cy='278.36' r='3' fill='blue'/><circle cx='696.25' cy='283.80' r='3' fill='blue'/><circle cx='745.00' cy='284.79' r='3' fill='blue'/><circle cx='793.75' cy='284.96' r='3' fill='blue'/><circle cx='842.50' cy='284.99' r='3' fill='blue'/><circle cx='891.25' cy='285.00' r='3' fill='blue'/><circle cx='940.00' cy='285.00' r='3' fill='blue'/><text x='745' y='55' text-anchor='middle'>Episode 2 (tigre droite)</text><text x='745' y='309' text-anchor='middle'>Ecoutes : 0 a 8</text><text x='542' y='70' text-anchor='end'>1.0</text><text x='542' y='180' text-anchor='end'>0.5</text><text x='542' y='285' text-anchor='end'>0.0</text><text x='555' y='83' fill='green'>Tigre reel</text><rect x='70' y='365' width='390' height='220' fill='none' stroke='black'/><line x1='70' x2='460' y1='475' y2='475' stroke='gray' stroke-dasharray='6 4'/><line x1='70' x2='460' y1='585.00' y2='585.00' stroke='green' stroke-dasharray='2 4'/><polyline points='70.00,475.00 118.75,552.00 167.50,578.36 216.25,583.80 265.00,584.79 313.75,584.96 362.50,584.99 411.25,585.00 460.00,585.00 ' fill='none' stroke='blue' stroke-width='2'/><circle cx='70.00' cy='475.00' r='3' fill='blue'/><circle cx='118.75' cy='552.00' r='3' fill='blue'/><circle cx='167.50' cy='578.36' r='3' fill='blue'/><circle cx='216.25' cy='583.80' r='3' fill='blue'/><circle cx='265.00' cy='584.79' r='3' fill='blue'/><circle cx='313.75' cy='584.96' r='3' fill='blue'/><circle cx='362.50' cy='584.99' r='3' fill='blue'/><circle cx='411.25' cy='585.00' r='3' fill='blue'/><circle cx='460.00' cy='585.00' r='3' fill='blue'/><text x='265' y='355' text-anchor='middle'>Episode 3 (tigre droite)</text><text x='265' y='609' text-anchor='middle'>Ecoutes : 0 a 8</text><text x='62' y='370' text-anchor='end'>1.0</text><text x='62' y='480' text-anchor='end'>0.5</text><text x='62' y='585' text-anchor='end'>0.0</text><text x='75' y='383' fill='green'>Tigre reel</text><rect x='550' y=

**Lecture de la figure — quatre trajectoires de belief, huit écoutes chacune.** Dans les sorties C#, les quatre panneaux montrent la même trajectoire : chaque panneau recrée un environnement avec la même graine. Le tigre est à droite et la croyance P(tiger-left) descend de 0.50 à 0.1500, puis 0.0302, avant de se rapprocher de zéro. La ligne verte indique la position réelle et la ligne grise le niveau de croyance initial de 0.5.

**En théorie**, une observation contradictoire peut inverser temporairement le mouvement de la croyance : par exemple, après deux observations à gauche, une observation à droite ramène la croyance de 0.9698 à 0.85. Cet exemple explique le filtre ; il ne décrit pas une variation visible dans les trajectoires C# affichées ici.

## 4. Q-MDP : Approximation du POMDP

Le **Q-MDP** (Littman et al., 1995) est une approximation classique pour les POMDPs :

1. **Entrainer** une Q-table sur les etats vrais (comme si l'environnement etait completement observable)
2. **Agir** en utilisant le belief : choisir l'action qui maximise $\sum_s b(s) \cdot Q(s, a)$

L'avantage : simple a implementer, reutilise le Q-learning standard.
L'inconvenient : ignore l'impact des actions sur les observations futures.

In [5]:
(double[,] Q, double mean, double[] returns) belief_qmdp(TigerPOMDP environnement, int n_episodes, int seed = 0)
{
    var rng = new Random(seed);
    var Q = new double[2, 3];
    var returns = new double[n_episodes];
    for (int ep = 0; ep < n_episodes; ep++)
    {
        environnement.reset();
        double belief = 0.5;
        double ret = 0.0;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            // Expected Q under belief
            int action;
            if (rng.NextDouble() < EPSILON)
                action = rng.Next(3);
            else
            {
                double[] expected = new double[3];
                for (int a = 0; a < 3; a++)
                    expected[a] = belief * Q[0, a] + (1.0 - belief) * Q[1, a];
                double maximum = expected.Max();
                int[] exaequo = Enumerable.Range(0, 3).Where(a => expected[a] == maximum).ToArray();
                action = exaequo[rng.Next(exaequo.Length)];
            }
            int old_true = environnement.tiger_pos;
            var transition = environnement.step(action);
            // Q-learning update with TRUE state
            double target = transition.reward;
            if (!transition.done)
            {
                int new_true = environnement.tiger_pos;
                double maximum = Math.Max(Q[new_true, 0], Math.Max(Q[new_true, 1], Q[new_true, 2]));
                target += GAMMA * maximum;
            }
            Q[old_true, action] += ALPHA * (target - Q[old_true, action]);
            // Belief update -- filtre couple a env.p_correct (voir cellule 2,
            // #25796) : bien specifie des que le capteur varie.
            if (action == 2)
                belief = mise_a_jour_belief(belief, transition.observation, environnement.p_correct);
            ret += transition.reward;
            if (transition.done) break;
        }
        returns[ep] = ret;
    }
    return (Q, returns.Skip(Math.Max(0, n_episodes - 200)).Average(), returns);
}

// Tolerances fixees avant l'entrainement : retour stochastique et Q-table apprises.
const double TOL_QMDP_RETURN = 25.0;
const double TOL_QMDP_Q = 5.0;
Console.WriteLine(FormattableString.Invariant($"Comparaison aux valeurs Python citees : tolerance retour {TOL_QMDP_RETURN:F1}, tolerance Q {TOL_QMDP_Q:F1} ; RNG C# distinct de NumPy."));
env = new TigerPOMDP();
var resultat_qmdp = belief_qmdp(env, 5000, seed: SEED);
var Q_qmdp = resultat_qmdp.Q;
var ret_qmdp = resultat_qmdp.mean;
var arr_qmdp = resultat_qmdp.returns;
Console.WriteLine("=== Q-MDP avec Belief Tracking ===");
Console.WriteLine(FormattableString.Invariant($"Return moyen (derniers 200 episodes): {ret_qmdp:F1}"));
Console.WriteLine("\nQ-table apprise:");
Console.WriteLine($"  {"Action",-15} {"Open-L",10} {"Open-R",10} {"Listen",10}");
Console.WriteLine(FormattableString.Invariant($"  {"Tiger-Left",-15} {Q_qmdp[0,0],10:F1} {Q_qmdp[0,1],10:F1} {Q_qmdp[0,2],10:F1}"));
Console.WriteLine(FormattableString.Invariant($"  {"Tiger-Right",-15} {Q_qmdp[1,0],10:F1} {Q_qmdp[1,1],10:F1} {Q_qmdp[1,2],10:F1}"));
Console.WriteLine("\nInterpretation:");
Console.WriteLine(FormattableString.Invariant($"  Si tiger-left: optimale = open-right (Q={Q_qmdp[0,1]:F1})"));
Console.WriteLine(FormattableString.Invariant($"  Si tiger-right: optimale = open-left (Q={Q_qmdp[1,0]:F1})"));
VerifierProche("Return Q-MDP ; Python cite", ret_qmdp, -10.5, TOL_QMDP_RETURN);
double[,] q_reference_python = { { -100.0, 10.0, 8.5 }, { 10.0, -100.0, 8.5 } };
for (int s = 0; s < 2; s++)
    for (int a = 0; a < 3; a++)
        VerifierProche($"Q[{s},{a}] ; Python cite", Q_qmdp[s, a], q_reference_python[s, a], TOL_QMDP_Q);
VerifierProche("Valeur theorique ecoute", -1.0 + GAMMA * 10.0, 8.5, 1e-12);

Comparaison aux valeurs Python citees : tolerance retour 25.0, tolerance Q 5.0 ; RNG C# distinct de NumPy.


=== Q-MDP avec Belief Tracking ===


Return moyen (derniers 200 episodes): -6.7



Q-table apprise:


  Action              Open-L     Open-R     Listen


  Tiger-Left          -100.0       10.0        8.5


... (8 outputs left out)

Q[1,0] ; Python cite : calcule=10.0000, attendu=10.0000, ecart=0.0000, tolerance=5.0000 : CONFORME


Q[1,1] ; Python cite : calcule=-100.0000, attendu=-100.0000, ecart=0.0000, tolerance=5.0000 : CONFORME


Q[1,2] ; Python cite : calcule=8.5000, attendu=8.5000, ecart=0.0000, tolerance=5.0000 : CONFORME


Valeur theorique ecoute : calcule=8.5000, attendu=8.5000, ecart=0.0000, tolerance=0.0000 : CONFORME


**Lecture chiffrée — la Q-table du Q-MDP.** Pour tigre à gauche, les valeurs apprises affichées en C# sont -100.0 pour ouvrir à gauche, 10.0 pour ouvrir à droite et 8.5 pour écouter ; la ligne opposée est symétrique. La valeur de l'écoute combine son coût immédiat et la valeur actualisée de la meilleure action dans l'état vrai. Pour ouvrir à droite, l'espérance 110b - 100 doit dépasser la valeur de l'écoute : deux observations concordantes conduisent à 0.9698, tandis qu'une observation concordante supplémentaire renforce encore la confiance.

Le retour sur les 200 derniers épisodes est **-6.7 dans ce run C#** ; la sortie Python citée est **-10.5**. Ces retours dépendent des tirages et incluent encore l'exploration.

### Pourquoi Q-MDP sous-performe-t-il ?

La Q-table apprise n'est pas en cause : `Listen = 8.5` (section precedente) signifie que la table a bien capture la valeur de l'information. Au moment d'agir, l'agent pondere cette table par son belief : tant que b < ~0.986, `Listen` domine l'esperance d'ouvrir (110 b - 100 < 8.5) et l'agent accumule des ecoutes — il n'est donc **pas** quasi-aleatoire en debut d'episode.

Ce qui lui manque est ailleurs : la Q-table evalue chaque action dans l'etat VRAI, comme si l'incertitude allait se resoudre gratuitement au pas suivant. Elle ne peut pas representer la SEQUENCE "encore une ecoute si le belief reste indecis, ouvrir sinon" — la valeur d'une ecoute depend du belief courant, pas de l'etat cache. C'est ce prix de sequence mal calcule (pas une decision aleatoire) qui explique l'ecart mesure face aux politiques expertes.

## 5. Belief-State Q-Learning

Une alternative plus directe : **discretiser le belief state** et apprendre une Q-table
directement dans cet espace.

Le belief du Tiger Problem est 1D : $b = P(\text{tiger-left}) \in [0, 1]$.
En discretisant en $N$ bins, on obtient un MDP avec $N$ etats, et le Q-learning standard s'applique.

In [6]:
(double[,] Q, double mean, double[] returns) belief_state_qlearning(TigerPOMDP environnement, int n_episodes, int n_belief_bins = 20, int seed = 0)
{
    var rng = new Random(seed);
    var Q = new double[n_belief_bins, 3];
    var returns = new double[n_episodes];
    for (int ep = 0; ep < n_episodes; ep++)
    {
        environnement.reset();
        double belief = 0.5;
        double ret = 0.0;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            int b_idx = Math.Min((int)(belief * n_belief_bins), n_belief_bins - 1);
            int action;
            if (rng.NextDouble() < EPSILON)
                action = rng.Next(3);
            else
            {
                double max = Math.Max(Q[b_idx, 0], Math.Max(Q[b_idx, 1], Q[b_idx, 2]));
                int[] choix = Enumerable.Range(0, 3).Where(a => Q[b_idx, a] == max).ToArray();
                action = choix[rng.Next(choix.Length)];
            }
            var transition = environnement.step(action);
            // Belief update -- filtre couple a env.p_correct (voir cellule 2,
            // #25796) : bien specifie des que le capteur varie.
            if (action == 2)
                belief = mise_a_jour_belief(belief, transition.observation, environnement.p_correct);
            int b_idx2 = Math.Min((int)(belief * n_belief_bins), n_belief_bins - 1);
            double target = transition.reward;
            if (!transition.done)
                target += GAMMA * Math.Max(Q[b_idx2, 0], Math.Max(Q[b_idx2, 1], Q[b_idx2, 2]));
            Q[b_idx, action] += ALPHA * (target - Q[b_idx, action]);
            ret += transition.reward;
            if (transition.done) break;
        }
        returns[ep] = ret;
    }
    return (Q, returns.Skip(Math.Max(0, n_episodes - 200)).Average(), returns);
}

// Tolerances declarees avant le calcul : les valeurs apprises dependent du RNG.
const double TOL_BS_RETURN = 30.0;
const double TOL_BS_Q = 35.0;
Console.WriteLine(FormattableString.Invariant($"Comparaison aux references Python citees : tolerance retour {TOL_BS_RETURN:F1}, tolerance Q {TOL_BS_Q:F1}."));
env = new TigerPOMDP();
var resultat_bs = belief_state_qlearning(env, 5000, n_belief_bins: 20, seed: SEED);
var Q_bs = resultat_bs.Q;
var ret_bs = resultat_bs.mean;
var arr_bs = resultat_bs.returns;
Console.WriteLine("=== Belief-State Q-Learning (20 bins) ===");
Console.WriteLine(FormattableString.Invariant($"Return moyen (derniers 200 episodes): {ret_bs:F1}"));
Console.WriteLine("\nQ-values par niveau de belief:");
Console.WriteLine($"  {"Belief P(TL)",-15} {"Open-L",10} {"Open-R",10} {"Listen",10}");
double[] niveaux_bs = { 0.05, 0.25, 0.50, 0.75, 0.95 };
double[,] reference_bs = { { 0, 0, 0 }, { 0, 0, 0 }, { -43.5, -43.7, 4.2 }, { 0, 0, 0 }, { -100, 8.2, -0.3 } };
for (int i = 0; i < niveaux_bs.Length; i++)
{
    int idx = Math.Min((int)(niveaux_bs[i] * 20), 19);
    Console.WriteLine(FormattableString.Invariant($"  {niveaux_bs[i],-15:F2} {Q_bs[idx,0],10:F1} {Q_bs[idx,1],10:F1} {Q_bs[idx,2],10:F1}"));
}
VerifierProche("Retour Belief Q ; Python cite", ret_bs, -4.0, TOL_BS_RETURN);
for (int i = 0; i < niveaux_bs.Length; i++)
{
    int idx = Math.Min((int)(niveaux_bs[i] * 20), 19);
    for (int a = 0; a < 3; a++)
        VerifierProche($"Belief {niveaux_bs[i]:F2}, action {a} ; Python cite", Q_bs[idx, a], reference_bs[i, a], i == 0 || i == 1 || i == 3 ? 1e-12 : TOL_BS_Q);
}

Comparaison aux references Python citees : tolerance retour 30.0, tolerance Q 35.0.


=== Belief-State Q-Learning (20 bins) ===


Return moyen (derniers 200 episodes): -4.7



Q-values par niveau de belief:


  Belief P(TL)        Open-L     Open-R     Listen


  0.05                   0.0        0.0        0.0


... (16 outputs left out)

Belief 0,75, action 2 ; Python cite : calcule=0.0000, attendu=0.0000, ecart=0.0000, tolerance=0.0000 : CONFORME


Belief 0,95, action 0 ; Python cite : calcule=-99.3640, attendu=-100.0000, ecart=0.6360, tolerance=35.0000 : CONFORME


Belief 0,95, action 1 ; Python cite : calcule=8.8665, attendu=8.2000, ecart=0.6665, tolerance=35.0000 : CONFORME


Belief 0,95, action 2 ; Python cite : calcule=-7.2703, attendu=-0.3000, ecart=6.9703, tolerance=35.0000 : CONFORME


**Lecture chiffree -- les zeros de la table sont la geometrie du belief.** Les lignes `0.05`, `0.25`, `0.75` affichent trois 0.0 chacune : ces bins ne sont jamais visites. La dynamique bayesienne a p = 0.85 ne produit qu'un echelier de valeurs -- 0.50, puis 0.85/0.15, puis 0.9698/0.0302, puis 0.9945/0.0055 -- qui atterrit dans les bins 10, 17/3, 19/0 du decoupage en 20 : aucun de ces indices n'est une ligne 0.05, 0.25 ou 0.75. La table non nulle ne vit que la ou le belief peut aller. Dans ce run C#, `Listen` vaut **2.9** au palier 0.50 mais **-7.3** a 0.95 : la table attribue des valeurs differentes a l'ecoute selon la croyance. Le retour de ce run est **-4.7**, superieur a celui de la politique experte a deux ecoutes mesuree en section 2 (**-9.2**). La comparaison sur cinq graines de la section suivante permet de replacer ce tirage unique dans une distribution de resultats.

## 6. Comparaison des méthodes

Le graphique ci-dessous resume les performances de toutes les approches testees.

In [7]:
// Tolerance fixee avant les simulations : comparaison aux statistiques Python citees.
const double TOL_COMPARAISON = 25.0;
Console.WriteLine(FormattableString.Invariant($"Tolerance des statistiques face aux references Python citees : {TOL_COMPARAISON:F1} points ; generateur C# distinct."));
env = new TigerPOMDP();
int[] SEEDS = { 0, 1, 7, 42, 99 };
var methods = new Dictionary<string, List<double>>
{
    ["Random"] = new List<double>(), ["Open immediat"] = new List<double>(),
    ["Listen x1"] = new List<double>(), ["Listen x2"] = new List<double>(),
    ["Q-MDP"] = new List<double>(), ["Belief Q"] = new List<double>()
};
foreach (int seed in SEEDS)
{
    methods["Random"].Add(random_agent(env, N_EP, seed: seed).mean);
    methods["Open immediat"].Add(open_immediately(env, N_EP, seed: seed).mean);
    methods["Listen x1"].Add(listen_then_open(env, N_EP, n_listen: 1, seed: seed).mean);
    methods["Listen x2"].Add(listen_then_open(env, N_EP, n_listen: 2, seed: seed).mean);
    methods["Q-MDP"].Add(belief_qmdp(env, N_EP, seed: seed).mean);
    methods["Belief Q"].Add(belief_state_qlearning(env, N_EP, n_belief_bins: 20, seed: seed).mean);
}
// Tableau resume
string[] noms_methodes = { "Random", "Open immediat", "Listen x1", "Listen x2", "Q-MDP", "Belief Q" };
double[,] reference_stats = { { -45.1, 0.6, -46.0, -44.3 }, { -45.6, 0.9, -47.2, -44.6 }, { -7.7, 0.9, -8.8, -6.4 }, { -8.0, 0.6, -9.1, -7.1 }, { -11.7, 4.2, -19.2, -7.4 }, { -10.6, 3.8, -13.1, -3.0 } };
var stats_methodes = new double[6, 4];
Console.WriteLine($"{"Methode",-20} {"Mean",8} {"Std",8} {"Min",8} {"Max",8}");
Console.WriteLine(new string('-', 54));
for (int i = 0; i < noms_methodes.Length; i++)
{
    var vals = methods[noms_methodes[i]];
    double moyenne = vals.Average();
    double ecart_type = Math.Sqrt(vals.Average(v => (v - moyenne) * (v - moyenne)));
    stats_methodes[i, 0] = moyenne;
    stats_methodes[i, 1] = ecart_type;
    stats_methodes[i, 2] = vals.Min();
    stats_methodes[i, 3] = vals.Max();
    Console.WriteLine(FormattableString.Invariant($"{noms_methodes[i],-20} {moyenne,8:F1} {ecart_type,8:F1} {vals.Min(),8:F1} {vals.Max(),8:F1}"));
}
for (int i = 0; i < noms_methodes.Length; i++)
    for (int j = 0; j < 4; j++)
        VerifierProche($"{noms_methodes[i]} statistique {j} ; Python cite", stats_methodes[i, j], reference_stats[i, j], TOL_COMPARAISON);

// Graphique
string[] couleurs = { "#e74c3c", "#f39c12", "#27ae60", "#2ecc71", "#3498db", "#9b59b6" };
var svg_comparaison = new StringBuilder();
svg_comparaison.Append("<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='500' viewBox='0 0 1000 500'><rect width='1000' height='500' fill='white'/><text x='500' y='25' text-anchor='middle' font-size='20'>Tiger Problem : Comparaison des methodes (5 seeds)</text>");
double haut = Math.Max(5.0, Enumerable.Range(0, 6).Max(i => stats_methodes[i, 0] + stats_methodes[i, 1]) + 3.0);
double bas = Math.Min(-55.0, Enumerable.Range(0, 6).Min(i => stats_methodes[i, 0] - stats_methodes[i, 1]) - 3.0);
Func<double, double> ygraph = valeur => 50.0 + (haut - valeur) / (haut - bas) * 350.0;
double zero = ygraph(0);
svg_comparaison.Append(FormattableString.Invariant($"<line x1='70' x2='970' y1='{zero:F2}' y2='{zero:F2}' stroke='gray'/><text x='15' y='210' transform='rotate(-90 15 210)'>Return moyen</text>"));
for (int i = 0; i < 6; i++)
{
    double x = 110 + i * 145;
    double y = ygraph(stats_methodes[i, 0]);
    double yhaut = ygraph(stats_methodes[i, 0] + stats_methodes[i, 1]);
    double ybas = ygraph(stats_methodes[i, 0] - stats_methodes[i, 1]);
    svg_comparaison.Append(FormattableString.Invariant($"<rect x='{x:F2}' y='{Math.Min(y, zero):F2}' width='75' height='{Math.Abs(y - zero):F2}' fill='{couleurs[i]}' stroke='black'/><line x1='{x + 37.5:F2}' x2='{x + 37.5:F2}' y1='{yhaut:F2}' y2='{ybas:F2}' stroke='black'/><line x1='{x + 30:F2}' x2='{x + 45:F2}' y1='{yhaut:F2}' y2='{yhaut:F2}' stroke='black'/><line x1='{x + 30:F2}' x2='{x + 45:F2}' y1='{ybas:F2}' y2='{ybas:F2}' stroke='black'/><text x='{x + 37.5:F2}' y='{y + 16:F2}' text-anchor='middle'>{stats_methodes[i,0]:F1}</text><text x='{x + 37.5:F2}' y='430' text-anchor='middle'>{noms_methodes[i]}</text>"));
}
svg_comparaison.Append("</svg>");
display(svg_comparaison.ToString(), "text/html");

Tolerance des statistiques face aux references Python citees : 25.0 points ; generateur C# distinct.


Methode                  Mean      Std      Min      Max


------------------------------------------------------


Random                  -45.2      0.6    -46.3    -44.6


Open immediat           -44.7      1.3    -46.2    -43.0


Listen x1                -7.8      0.4     -8.2     -7.2


... (24 outputs left out)

Belief Q statistique 1 ; Python cite : calcule=2.3443, attendu=3.8000, ecart=1.4557, tolerance=25.0000 : CONFORME


Belief Q statistique 2 ; Python cite : calcule=-10.2400, attendu=-13.1000, ecart=2.8600, tolerance=25.0000 : CONFORME


Belief Q statistique 3 ; Python cite : calcule=-3.1500, attendu=-3.0000, ecart=0.1500, tolerance=25.0000 : CONFORME


<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='500' viewBox='0 0 1000 500'><rect width='1000' height='500' fill='white'/><text x='500' y='25' text-anchor='middle' font-size='20'>Tiger Problem : Comparaison des methodes (5 seeds)</text><line x1='70' x2='970' y1='79.17' y2='79.17' stroke='gray'/><text x='15' y='210' transform='rotate(-90 15 210)'>Return moyen</text><rect x='110.00' y='79.17' width='75' height='263.94' fill='#e74c3c' stroke='black'/><line x1='147.50' x2='147.50' y1='339.74' y2='346.48' stroke='black'/><line x1='140.00' x2='155.00' y1='339.74' y2='339.74' stroke='black'/><line x1='140.00' x2='155.00' y1='346.48' y2='346.48' stroke='black'/><text x='147.50' y='359.11' text-anchor='middle'>-45.2</text><text x='147.50' y='430' text-anchor='middle'>Random</text><rect x='255.00' y='79.17' width='75' height='260.83' fill='#f39c12' stroke='black'/><line x1='292.50' x2='292.50' y1='332.48' y2='347.52' stroke='black'/><line x1='285.00' x2='300.00' y1='332.48' y2='332.48' stroke='black'/><line x1='285.00' x2='300.00' y1='347.52' y2='347.52' stroke='black'/><text x='292.50' y='356.00' text-anchor='middle'>-44.7</text><text x='292.50' y='430' text-anchor='middle'>Open immediat</text><rect x='400.00' y='79.17' width='75' height='45.76' fill='#27ae60' stroke='black'/><line x1='437.50' x2='437.50' y1='122.71' y2='127.14' stroke='black'/><line x1='430.00' x2='445.00' y1='122.71' y2='122.71' stroke='black'/><line x1='430.00' x2='445.00' y1='127.14' y2='127.14' stroke='black'/><text x='437.50' y='140.93' text-anchor='middle'>-7.8</text><text x='437.50' y='430' text-anchor='middle'>Listen x1</text><rect x='545.00' y='79.17' width='75' height='51.12' fill='#2ecc71' stroke='black'/><line x1='582.50' x2='582.50' y1='127.84' y2='132.74' stroke='black'/><line x1='575.00' x2='590.00' y1='127.84' y2='127.84' stroke='black'/><line x1='575.00' x2='590.00' y1='132.74' y2='132.74' stroke='black'/><text x='582.50' y='146.29' text-anchor='middle'>-8.8</text><text x='582.50' y='430' text-anchor='middle'>Listen x2</text><rect x='690.00' y='79.17' width='75' height='75.31' fill='#3498db' stroke='black'/><line x1='727.50' x2='727.50' y1='138.22' y2='170.73' stroke='black'/><line x1='720.00' x2='735.00' y1='138.22' y2='138.22' stroke='black'/><line x1='720.00' x2='735.00' y1='170.73' y2='170.73' stroke='black'/><text x='727.50' y='170.47' text-anchor='middle'>-12.9</text><text x='727.50' y='430' text-anchor='middle'>Q-MDP</text><rect x='835.00' y='79.17' width='75' height='39.33' fill='#9b59b6' stroke='black'/><line x1='872.50' x2='872.50' y1='104.82' y2='132.17' stroke='black'/><line x1='865.00' x2='880.00' y1='104.82' y2='104.82' stroke='black'/><line x1='865.00' x2='880.00' y1='132.17' y2='132.17' stroke='black'/><text x='872.50' y='134.50' text-anchor='middle'>-6.7</text><text x='872.50' y='430' text-anchor='middle'>Belief Q</text></svg>

**Lecture chiffrée — cinq graines.** La table C# précédente donne Listen x1 **-7.8 ± 0.4**, Listen x2 **-8.8 ± 0.4**, Belief Q **-6.7 ± 2.3**, Q-MDP **-12.9 ± 2.8**, Random **-45.2 ± 0.6** et ouverture immédiate **-44.7 ± 1.3**. Dans ces tirages, Belief Q devance les deux politiques à nombre fixe d'écoutes, tandis que Q-MDP les suit.

À titre de comparaison, les **sorties Python citées** donnent respectivement -7.7 ± 0.9, -8.0 ± 0.6, -10.6 ± 3.8, -11.7 ± 4.2, -45.1 ± 0.6 et -45.6 ± 0.9 : le classement de Belief Q diffère. Les générateurs sont distincts. De plus, cette table compare des politiques fixes exécutées sans exploration aux 200 derniers épisodes d'entraînement des méthodes apprenantes, qui conservent epsilon = 0.1. Le contrôle suivant examine leurs politiques gloutonnes.

In [8]:
// Controle apparie : evaluation gloutonne (epsilon=0) des politiques apprenantes.
// La table precedente compare les politiques fixes sans exploration aux 200
// derniers episodes d'entrainement des methodes apprenantes. Ici, les politiques
// apprises sont evaluees gloutonnement sur 200 episodes frais. Les filtres
// utilisent la precision reelle du capteur de chaque environnement.

double eval_greedy_beliefq(TigerPOMDP environnement, double[,] Q, int n_episodes = 200, int n_belief_bins = 20)
{
    double somme = 0;
    for (int ep = 0; ep < n_episodes; ep++)
    {
        environnement.reset();
        double belief = 0.5;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            int idx = Math.Min((int)(belief * n_belief_bins), n_belief_bins - 1);
            int action = 0;
            for (int a = 1; a < 3; a++) if (Q[idx, a] > Q[idx, action]) action = a;
            var transition = environnement.step(action);
            somme += transition.reward;
            if (action == 2) belief = mise_a_jour_belief(belief, transition.observation, environnement.p_correct);
            if (transition.done) break;
        }
    }
    return somme / n_episodes;
}

double eval_greedy_qmdp(TigerPOMDP environnement, double[,] Q, int n_episodes = 200)
{
    double somme = 0;
    for (int ep = 0; ep < n_episodes; ep++)
    {
        environnement.reset();
        double belief = 0.5;
        for (int t = 0; t < MAX_STEPS; t++)
        {
            double[] expected = new double[3];
            for (int a = 0; a < 3; a++) expected[a] = belief * Q[0, a] + (1 - belief) * Q[1, a];
            int action = 0;
            for (int a = 1; a < 3; a++) if (expected[a] > expected[action]) action = a;
            var transition = environnement.step(action);
            somme += transition.reward;
            if (action == 2) belief = mise_a_jour_belief(belief, transition.observation, environnement.p_correct);
            if (transition.done) break;
        }
    }
    return somme / n_episodes;
}

double valeur_k_fixe(double p, int k)
{
    double correct = 0;
    double combinaison = 1;
    for (int j = 0; j <= k; j++)
    {
        double a = Math.Pow(p, j) * Math.Pow(1 - p, k - j);
        double b = Math.Pow(1 - p, j) * Math.Pow(p, k - j);
        double poids = combinaison * a;
        double post = a / (a + b);
        correct += poids * Math.Max(post, 1 - post);
        if (j < k) combinaison *= (double)(k - j) / (j + 1);
    }
    return correct * 10 + (1 - correct) * -100 - k;
}

double valeur_seuil(double p, int tau)
{
    double r = (1 - p) / p;
    double correct = (1 - Math.Pow(r, tau)) / (1 - Math.Pow(r, 2 * tau));
    double ecoutes = (2 * correct - 1) * tau / (2 * p - 1);
    return 10 * correct - 100 * (1 - correct) - ecoutes;
}

const int N_EP_CTRL = 3000;
double[] P_GRID = { 0.85, 0.75, 0.70, 0.65, 0.60 };
// Tolerance declaree avant simulation pour les moyennes stochastiques citees.
const double TOL_CTRL = 25.0;
const double TOL_THEORIE = 0.051;
Console.WriteLine(FormattableString.Invariant($"Tolerance controle stochastique : {TOL_CTRL:F1} ; references theoriques arrondies : {TOL_THEORIE:F3}."));
var controle = new double[5, 3];
double[,] reference_ctrl = { { -7.9, -1.9, 4.6 }, { -18.4, -2.5, -1.7 }, { -24.0, -5.2, -7.0 }, { -29.7, -9.1, -11.7 }, { -35.1, -16.2, -18.6 } };
Console.WriteLine("Controle apparie : meme budget d'entrainement, evaluation gloutonne (200 episodes)");
Console.WriteLine("p_correct |  Listen x1   Belief Q      Q-MDP");
Console.WriteLine(new string('-', 48));
for (int pi = 0; pi < P_GRID.Length; pi++)
{
    double p = P_GRID[pi];
    var vals_l1 = new List<double>();
    var vals_bq = new List<double>();
    var vals_qm = new List<double>();
    foreach (int seed in SEEDS)
    {
        vals_l1.Add(listen_then_open(new TigerPOMDP(p_correct: p), N_EP_CTRL, n_listen: 1, seed: seed).mean);
        var qb = belief_state_qlearning(new TigerPOMDP(p_correct: p), N_EP_CTRL, n_belief_bins: 20, seed: seed).Q;
        vals_bq.Add(eval_greedy_beliefq(new TigerPOMDP(p_correct: p), qb));
        var qm = belief_qmdp(new TigerPOMDP(p_correct: p), N_EP_CTRL, seed: seed).Q;
        vals_qm.Add(eval_greedy_qmdp(new TigerPOMDP(p_correct: p), qm));
    }
    controle[pi, 0] = vals_l1.Average();
    controle[pi, 1] = vals_bq.Average();
    controle[pi, 2] = vals_qm.Average();
    Console.WriteLine(FormattableString.Invariant($"{p,9:F2} | {controle[pi,0],10:F1} {controle[pi,1],10:F1} {controle[pi,2],10:F1}"));
}
for (int pi = 0; pi < P_GRID.Length; pi++)
    for (int j = 0; j < 3; j++)
        VerifierProche($"Controle p={P_GRID[pi]:F2}, methode {j} ; Python cite", controle[pi, j], reference_ctrl[pi, j], TOL_CTRL);
Console.WriteLine("\nReferences theoriques (esperances exactes) :");
double[] reference_fixe = { 2.1, -4.4, -9.6, -17.4, -28.1 };
double[] reference_seuil = { 5.2, 0.9, -2.9, -9.8, -21.5 };
int[] reference_k = { 5, 9, 11, 11, 11 };
int[] reference_tau = { 3, 4, 4, 4, 4 };
for (int pi = 0; pi < P_GRID.Length; pi++)
{
    double p = P_GRID[pi];
    int k_best = Enumerable.Range(1, 12).OrderByDescending(k => valeur_k_fixe(p, k)).First();
    int tau_best = Enumerable.Range(1, 12).OrderByDescending(t => valeur_seuil(p, t)).First();
    double fixe = valeur_k_fixe(p, k_best), seuil = valeur_seuil(p, tau_best);
    Console.WriteLine(FormattableString.Invariant($"  p_correct = {p:F2} : meilleur k fixe {fixe:+0.0;-0.0;0.0} (k={k_best}) ; seuil adaptatif {seuil:+0.0;-0.0;0.0} (tau={tau_best})"));
    VerifierProche($"Reference fixe p={p:F2}", fixe, reference_fixe[pi], TOL_THEORIE);
    VerifierProche($"Reference seuil p={p:F2}", seuil, reference_seuil[pi], TOL_THEORIE);
    if (k_best != reference_k[pi] || tau_best != reference_tau[pi]) throw new InvalidOperationException("Optimum theorique incorrect");
}
double ecart_entrainement = stats_methodes[4, 0] - stats_methodes[2, 0];
double ecart_glouton = controle[0, 2] - controle[0, 0];
double ecart_85 = controle[0, 1] - controle[0, 2];
double ecart_70 = controle[2, 1] - controle[2, 2];
Console.WriteLine(FormattableString.Invariant($"Inversion protocole : Q-MDP moins Listen x1 en entrainement = {ecart_entrainement:F2} ; en evaluation gloutonne a 0.85 = {ecart_glouton:F2}"));
Console.WriteLine(FormattableString.Invariant($"Inversion bruit : Belief Q moins Q-MDP a 0.85 = {ecart_85:F2} ; a 0.70 = {ecart_70:F2}"));
if (!(ecart_entrainement < 0 && ecart_glouton > 0))
    throw new InvalidOperationException("Inversion de protocole absente");
if (!(ecart_85 < 0 && ecart_70 > 0))
    throw new InvalidOperationException("Inversion du classement selon le bruit absente");

Tolerance controle stochastique : 25.0 ; references theoriques arrondies : 0.051.


Controle apparie : meme budget d'entrainement, evaluation gloutonne (200 episodes)


p_correct |  Listen x1   Belief Q      Q-MDP


------------------------------------------------


     0.85 |       -8.3        4.0        4.5


     0.75 |      -19.2       -1.8       -0.2


... (32 outputs left out)

Reference fixe p=0,60 : calcule=-28.1152, attendu=-28.1000, ecart=0.0152, tolerance=0.0510 : CONFORME


Reference seuil p=0,60 : calcule=-21.5464, attendu=-21.5000, ecart=0.0464, tolerance=0.0510 : CONFORME


Inversion protocole : Q-MDP moins Listen x1 en entrainement = -5.07 ; en evaluation gloutonne a 0.85 = 12.85


Inversion bruit : Belief Q moins Q-MDP a 0.85 = -0.58 ; a 0.70 = 1.85


### Lecture du contrôle apparié — deux inversions successives

À 85 %, le contrôle C# donne **-8.3** pour Listen x1, **+4.0** pour Belief Q et **+4.5** pour Q-MDP. Dans la table d'entraînement précédente, Q-MDP était derrière Listen x1 ; en évaluation gloutonne, il passe devant. À 85 %, Q-MDP glouton devance Belief Q, tandis qu'à 70 % Belief Q (**-3.2**) devance Q-MDP (**-5.0**) : le classement entre les deux méthodes s'inverse lorsque le capteur devient plus bruité.

Les **sorties Python citées**, issues d'un générateur distinct, donnent à 85 % -7.9, -1.9 et +4.6 respectivement, puis à 70 % -5.2 pour Belief Q et -7.0 pour Q-MDP. Elles montrent les mêmes inversions sans reproduire les décimales C#.

Les références théoriques calculées dans la cellule précédente concernent d'autres familles de politiques : à 85 %, le meilleur nombre fixe d'écoutes vaut environ +2.1 et le seuil adaptatif environ +5.2. Elles ne tiennent pas compte de la même manière de la limite de 20 pas du simulateur. Le filtre utilise la précision réelle du capteur : il reste ainsi adapté lorsque celle-ci varie.

### Résultats clés

1. **Écouter paie** : ouvrir sans information vaut exactement -45 en espérance ; une écoute à 85 % donne -7.5, coût compris.
2. **Le belief tracking** maintient une distribution sur l'état caché. Dans le contrôle C#, les deux méthodes apprenantes gloutonnes dépassent Listen x1 aux précisions testées ; à 85 %, Q-MDP atteint +4.5. La référence Python citée pour cette dernière valeur est +4.6.
3. **État connu contre état caché** : connaître la position du tigre permettrait de gagner +10 en ouvrant directement la bonne porte. Les observations successives réduisent l'écart dû à l'observabilité partielle.

## 7. Des POMDP au RL moderne

Les POMDPs sont partout en RL applique :

| POMDP | RL moderne |
|-------|------------|
| Etat cache | Etat du marche, intentions d'autres agents |
| Observation bruitee | Capteurs, images, texte |
| Belief state | RNN hidden state, transformer context |
| Q-MDP approximation | DRQN (Deep Recurrent Q-Network) |

### DRQN (Hausknecht & Stone, 2015)

Le **Deep Recurrent Q-Network** remplace le belief tracking manuel par un **RNN** (LSTM/GRU)
qui apprend implicitement a maintenir un belief state a partir de l'historique des observations.

### PPO + LSTM (notebook 6c)

Dans les environnements partiellement observables, PPO utilise souvent un reseau avec
memoire (LSTM) pour integrer les observations passees — exactement le rôle du belief tracker.

### AlphaGo et la théorie des jeux

Le jeu de Go est un POMDP : un joueur ne connait pas les intentions de son adversaire.
AlphaGo utilise un **reseau de politique** qui encode un "belief" sur les coups adverses.

## 8. Exercices

### Exercice 1 : Impact de la precision d'observation

Faites varier `p_correct` entre 0.5 (aleatoire) et 1.0 (parfait) et tracez le return
de la politique "open immediately" (porte au hasard, aucune information) et de
"listen x2". A quel seuil la politique "open immediately" devient-elle meilleure
que "listen x2" ? Pourquoi ce seuil est-il si proche de p = 0.5 ?

In [9]:
// Exercice 1 : Impact de la precision d'observation
// TODO : faites varier p_correct de 0.5 a 1.0 et tracez les courbes
// Hint: utilisez np.linspace(0.5, 1.0, 11) pour les valeurs de p_correct

var precisions = new List<double>(); // TODO etudiant : liste de precisions
var results_imm = new List<double>(); // TODO etudiant : returns pour open_immediately
var results_l2 = new List<double>(); // TODO etudiant : returns pour listen x2

object result_ex1 = null; // TODO etudiant
Console.WriteLine("Exercice a completer : tracez return vs precision");

Exercice a completer : tracez return vs precision


**Lecture du stub — exercice 1, le seuil de precision.** `Exercice a completer : tracez return vs precision`. Les deux courbes n'ont pas la meme forme : "open immediately" ouvre sans aucune information, son esperance est la constante (10 - 100)/2 = -45, une droite horizontale sur tout le balayage ; "listen x2" suit la droite 110 p - 102 (deux ecoutes payees) — nulle vers p ~ 0.93, egale a -45 exactement quand 110 p = 57, soit p = 0.518. Le croisement demande vit donc tout pres de p = 0.5 : sous ce seuil, la deuxieme ecoute ne transporte pas assez d'information pour rembourser son cout de -1, et ouvrir a l'aveugle fait mieux ; au-dessus, chaque point de precision gagne 110 d'esperance pour la politique qui ecoute, contre 0 pour l'aveugle. C'est la valeur d'information, rendue visible par un seuil.

### Exercice 2 : Politique optimale du nombre d'ecoutes

Trouvez le nombre optimal d'ecoutes $N^*$ pour la politique "listen $N$ fois puis ouvre"
en fonction de $P(\text{correct})$. Verifiez que $N^*(0.85) = 5$ (esperance ~ +2.1), et
expliquez pourquoi un nombre PAIR d'ecoutes sous-performe systematiquement son voisin
impair (indice : que fait `argmin` sur un belief exactement egal a 0.5 ?). Que valent
$N^*(0.5)$ et $N^*(1.0)$ ?

In [10]:
// Exercice 2 : Nombre optimal d'ecoutes
// TODO : testez n_listen de 0 a 5 pour differentes precisions
// Hint: pour chaque (precision, n_listen), executez listen_then_open

object result_ex2 = null; // TODO etudiant
Console.WriteLine("Exercice a completer : trouvez N* en fonction de P");

Exercice a completer : trouvez N* en fonction de P


**Lecture du stub — exercice 2, le nombre optimal d'ecoutes.** `Exercice a completer : trouvez N* en fonction de P`. Les paliers de la section 3 donnent la forme du terrain : chaque ecoute concordante resserre le belief (0.50 -> 0.85 -> 0.9698 -> 0.9945 a p = 0.85) mais le gain marginal s'aplatit vite — la troisieme ecoute ne fait gagner que ~0.025 de confiance pour -1. Les regimes limites sont les deux bouts du balayage : a p = 1.0, une seule ecoute suffit a atteindre la certitude ; a p = 0.5, chaque ecoute est independante de l'etat, ne transporte AUCUNE information (la lecture de l'exercice 1 l'a deja dit) et coute -1 — dans la famille "listen N fois puis ouvre", la reponse a mesurer n'est donc pas forcement "plus d'ecoutes", et la valeur N* = 0 est une candidate a verifier. Entre les deux extremes, la courbe N*(p) a tracer n'a aucune raison d'etre monotone : c'est precisement ce que la mesure doit etablir, point par point, plutot qu'une forme imposee a l'avance. La section 2 fournit les trois politiques mesurees a p = 0.85 — un point sur la courbe, pas la courbe.

### Exercice 3 : Ajouter une troisieme porte

Etendez le Tiger Problem a 3 portes (tiger derriere l'une, tresor derriere les deux autres).
Combien d'etats, d'actions, d'observations ? Le belief tracking change-t-il fondamentalement ?

In [11]:
// Exercice 3 : Tiger Problem a 3 portes
// TODO : creez Tiger3DoorsPOMDP et testez les politiques
// Hint: 3 etats, 4 actions (open-L, open-M, open-R, listen), 3 observations

class Tiger3DoorsPOMDP
{
    // TODO etudiant
}

object result_ex3 = null; // TODO etudiant
Console.WriteLine("Exercice a completer : Tiger Problem a 3 portes");

Exercice a completer : Tiger Problem a 3 portes


## À retenir

Le Tiger Problem illustre la décision sous observation partielle : écouter coûte -1 mais fournit une observation correcte à 85 %, tandis qu'une mauvaise porte coûte -100 et la bonne rapporte +10. Le filtre bayésien maintient une croyance sur la position cachée ; Q-MDP combine cette croyance avec une table apprise sur les états vrais, tandis que Belief Q apprend directement sur des croyances discrétisées.

Dans le tirage C#, deux observations à droite conduisent de 0.50 à 0.0302. La trajectoire Python citée dans la section 3 atteint 0.9698 après quatre observations différentes. Les générateurs distincts expliquent ces trajectoires différentes.

Sur cinq graines, la table C# donne notamment Belief Q **-6.7 ± 2.3**, Q-MDP **-12.9 ± 2.8** et Listen x1 **-7.8 ± 0.4**. Les chiffres Python cités sont respectivement **-10.6 ± 3.8**, **-11.7 ± 4.2** et **-7.7 ± 0.9**. Cette comparaison mélange exécution des politiques fixes et fin d'entraînement avec exploration ; le contrôle glouton distingue les protocoles et montre les deux inversions décrites en section 6.

## Vérifiez votre compréhension

1. Pourquoi une écoute peut-elle valoir son coût malgré une probabilité d'erreur de 15 % ?

<details>
<summary>Réponse</summary>

Une ouverture aveugle rapporte en espérance -45. Une écoute coûte -1, mais permet ensuite de choisir la porte opposée à l'observation : à 85 % de précision, le retour attendu est -7.5. L'information réduit donc suffisamment le risque d'ouvrir sur le tigre pour compenser son coût.
</details>

2. Comment une observation contradictoire modifie-t-elle la croyance ?

<details>
<summary>Réponse</summary>

Le filtre multiplie chaque croyance par la vraisemblance de l'observation, puis normalise. Une observation à droite après une observation à gauche ramène par exemple la croyance de 0.85 à 0.50. Dans le tirage C# affiché, les deux observations sont au contraire concordantes et conduisent à 0.0302 pour la probabilité que le tigre soit à gauche.
</details>

3. Pourquoi distinguer retour d'entraînement et évaluation gloutonne ?

<details>
<summary>Réponse</summary>

Pendant l'entraînement, epsilon-greedy choisit parfois une action aléatoire, ce qui peut entraîner une ouverture coûteuse. L'évaluation gloutonne mesure la politique apprise sans cette exploration. Dans les sorties C#, Q-MDP est derrière Listen x1 dans la table d'entraînement sur cinq graines, mais passe devant dans le contrôle glouton à 85 % ; les deux protocoles ne mesurent donc pas exactement la même chose.
</details>

## Pour aller plus loin

- **[Notebook 12 — Distributional RL (C51)](RL-12-Distributional-RL-C51-Python.ipynb)** : modéliser la distribution du retour.
- **Notebook 5** : fondements MDP et Q-learning.
- **Notebook 8** : RL fondé sur un modèle et planification.
- **Notebook 10** : reward shaping et curriculum learning.
- **[DecInfer-07 — Sequential : MDPs, bandits, POMDPs](../Probas/DecisionTheory/DecInfer/DecInfer-07-Sequential.ipynb)** : observation partielle et décision.
- **[DecInfer-07b — Gittins (Lean)](../Probas/DecisionTheory/DecInfer/DecInfer-07b-Lean-Gittins.ipynb)** : arrêt optimal de la collecte d'information.
- **Jumeau Python** → [RL-11-POMDP-Croyances-Python](RL-11-POMDP-Croyances-Python.ipynb) — le carnet d'origine, en Python.

### Lectures

1. Cassandra, Littman & Kaelbling (1994), *Acting optimally in partially observable stochastic domains*.
2. Littman, Cassandra & Kaelbling (1995), *Learning policies for partially observable environments*.
3. Hausknecht & Stone (2015), *Deep recurrent Q-learning for partially observable MDPs*.
4. Oliehoek & Amato (2016), *A concise introduction to decentralized POMDPs*.
5. Thrun, Burgard & Fox (2005), *Probabilistic Robotics*.